# WSJ 2027 - Gruppindelning IST (rundresa)

Group the IST functionaries who travel via the rundresa bus into transport
groups. Same algorithm as deltagare, but:
- IST registration form doesn't ask for friend wishes — friend optimisation
  has nothing to work with, so the result is driven purely by geography,
  kår diversity and parent-org/age/sex balance.
- Group size and max_kar match the deltagare config so IST and deltagare
  groups have comparable density on the buses.
- IST_EGEN_RESA travel on their own and are NOT included here.

## Rules
1. **Around 40 per group (3 full + 1 remainder; matches IST bus capacity)** (remainder group allowed)
2. **Geographic proximity** — Hilbert curve sort, kår-cluster aware
3. **Kår-konsolation** — alla med en kår (≥2 medlemmar på resan) ska helst ha en kår-kompis i gruppen (soft goal; Phase 3.5 + Phase 4 SA). Extra viktig för IST eftersom det inte finns några vän-önskningar att luta sig mot.
4. **Max 6 from same kår** per group (hard constraint)
5. **Diversity** — age and sex balanced, parent-org mixed (Phase 4 SA)


In [ ]:
TRAVEL = 'rundresa'        # only line that differs between rundresa and direktresa
QUALITY = 'slow'           # 'medium' or 'slow' (8 restarts)
WEIGHT_PROFILE = 'friend_geo'  # 'balanced' or 'friend_geo'
COORD_SOURCE = 'home'      # 'kar' (kår-first) or 'home' (home-address-first)
GROUP_SIZE = 40
MAX_KAR = 6
OUTPUT_DIR = '/config/notebooks/wsj27/output'


In [ ]:
import sys; sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

raw = u.fetch_participants()
df_all, _ = u.build_participant_dataframe(raw)
df = df_all[(df_all['travel'] == TRAVEL) & (df_all['category'] == 'ist')].copy().reset_index(drop=True)
u.assign_coordinates(df, coord_source=COORD_SOURCE)
df = u.add_hilbert_index(df)
u.resolve_friend_wishes(df, df_all)
u.apply_manual_overrides(df, df_all)
friend_wishes = u.build_friend_graph(df)
u.print_intake_summary(df, GROUP_SIZE)


In [ ]:
df = u.assign_groups(df, GROUP_SIZE, friend_wishes,
                     max_kar=MAX_KAR, quality=QUALITY,
                     weight_profile=WEIGHT_PROFILE)
u.apply_manual_placement(df)
group_of = df['group'].values
total_groups = df['group'].nunique()


In [ ]:
u.print_group_metrics(df, group_of, total_groups)
csv_path, json_path = u.export_results(df, group_of, total_groups, OUTPUT_DIR, prefix=f'wsj27_ist_{TRAVEL}')
map_path = f'{OUTPUT_DIR}/wsj_ist_{TRAVEL}_karta.html'
u.generate_group_map_html(df, total_groups, map_path,
                          title=f'WSJ 2027 IST {TRAVEL.title()}',
                          friend_wishes=friend_wishes, show_group_arcs=True)
report_path = f'{OUTPUT_DIR}/wsj_ist_{TRAVEL}_grupper.html'
u.generate_groups_report_html(df, total_groups, report_path,
                              title=f'WSJ 2027 IST {TRAVEL.title()} - Grupper',
                              friend_wishes=friend_wishes,
                              group_size=GROUP_SIZE,
                              max_kar=MAX_KAR,
                              quality=QUALITY,
                              weight_profile=WEIGHT_PROFILE,
                              coord_source=COORD_SOURCE)
print(f'CSV:    {csv_path}\nJSON:   {json_path}\nMap:    {map_path}\nReport: {report_path}')
